# Synthetic student journal RL model
This notebook trains a model to simulate a Fontys ICT student's psychological state using reinforcent learning techiniques.

### Observations / environment (the student's internal state)
These are the variables the env tracks at each timestep (probably = one simulated day):

- `energy (0–1)` — depletes through cognitive load, restores through sleep/rest. Ego depletion lives here.
- `stress (0–1)` — rises with deadlines/workload, falls with rest or completed tasks.
- `self_efficacy (0–1)` — slowly shifts based on recent successes or failures.
- `social_need (0–1)` — rises over time if not socializing, drops after social activity.
- `days_until_deadline (int)` — a simple external pressure signal.

### Actions (what the agent can do each day)
- `study`
- `attend lecture`
- `leisure`
- `socialize`
- `rest`
- `skip / do nothing`

### Reward (What should the agent work towards)
The reward function is not yet finalized — this is a placeholder to get the environment running. For now we use reward = self_efficacy - stress, which loosely optimizes for a student who feels capable and not overwhelmed. This may be revised once we have a clearer picture of what the model should actually be learning to do.

---

## Changes to the environment

### General rules that run every step regardless of action:
- days_until_deadline ticks down by 1
- stress increases passively if deadline is close: stress += 0.1 * (1 - days_until_deadline/max_days)
- social_need drifts up passively each day: social_need = min(1, social_need + 0.05)

### Per action:
**study**
- energy -= 0.2 (cognitive load)
- stress -= 0.15 (making progress)
- self_efficacy += 0.05 (competence building)

**attend lecture**
- energy -= 0.15 (less than studying, more passive)
- stress -= 0.05 (slight relief, staying on track)
- self_efficacy += 0.03 (competence, but slower than active studying)

**leisure**
- energy += 0.15 (restores somewhat, less than full rest)
- stress -= 0.15 
- social_need -= 0.1
- self_efficacy -= 0.01 (very slight guilt/opportunity cost, barely noticeable)

**socialize**
- energy -= 0.1
- social_need -= 0.3 
- self_efficacy += 0.02 (relatedness boost, small)

**rest**
- energy += 0.3
- stress -= 0.1

**skip / do nothing**
- energy += 0.1 (short term relief)
- stress += 0.05 (guilt/falling behind)
- self_efficacy -= 0.05

---

## Dependencies

In [19]:
import gymnasium as gym
import numpy as np

print("Gymnasium version:", gym.__version__)
print("NumPy version:", np.__version__)

Gymnasium version: 1.3.0
NumPy version: 1.26.4


## Intialize the environment

In [20]:
class StudentEnv(gym.Env):
    def __init__(self):
        super(StudentEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(6)  # Study, Attend lecture, Leisure, Socialize, Rest, Skip
        
        # Observation: [energy, stress, self_efficacy, social_need, days_until_deadline]
        self.observation_space = gym.spaces.Box(
            low=0, 
            high=1, 
            shape=(5,), 
            dtype=np.float32
        )
        
        # Initialize state
        self.state = np.array([
            0.5,   # energy
            0.5,   # stress
            0.5,   # self_efficacy
            0.5,   # social_need
            10.0   # days_until_deadline (not 0-1, it's an int conceptually)
        ], dtype=np.float32)
        
        self.timestep = 0
        self.max_days = 30

    def step(self, action):
        energy, stress, self_efficacy, social_need, _ = self.state  # ignore normalized days, use internal
        
        # Passive ticks
        stress += 0.1 * (1 - self.days_until_deadline / self.max_days)
        social_need = min(1, social_need + 0.05)
        self.days_until_deadline = max(0, self.days_until_deadline - 1)
        
        if self.days_until_deadline == 0:
            self.days_until_deadline = 10  # reset to new deadline
            stress = min(1, stress + 0.2)  # deadline passing costs you

        # Per-action effects
        if action == 0:  # Study
            if stress > 0.8:
                energy -= 0.3
                stress += 0.05
                self_efficacy -= 0.05
            else:
                energy -= 0.2
                stress -= 0.15
                self_efficacy += 0.05
        elif action == 1:  # Attend lecture
            energy -= 0.15
            stress -= 0.05
            self_efficacy += 0.03
        elif action == 2:  # Leisure
            energy += 0.15
            stress -= 0.15
            social_need -= 0.1
            self_efficacy -= 0.01
        elif action == 3:  # Socialize
            energy -= 0.1
            social_need -= 0.3
            self_efficacy += 0.02
        elif action == 4:  # Rest
            energy += 0.3
            stress -= 0.1
        elif action == 5:  # Skip
            energy += 0.1
            stress += 0.05
            self_efficacy -= 0.05

        # Clip all to [0, 1]
        energy = np.clip(energy, 0, 1)
        stress = np.clip(stress, 0, 1)
        self_efficacy = np.clip(self_efficacy, 0, 1)
        social_need = np.clip(social_need, 0, 1)

        self.state = np.array([
            energy,
            stress,
            self_efficacy,
            social_need,
            self.days_until_deadline / self.max_days  # normalized here
        ], dtype=np.float32)

        reward = self_efficacy - stress
        terminated = False
        truncated = False

        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_until_deadline = 10  # raw int lives here now
        self.state = np.array([
            0.5,  # energy
            0.5,  # stress
            0.5,  # self_efficacy
            0.5,  # social_need
            self.days_until_deadline / self.max_days  # normalized
        ], dtype=np.float32)
        self.timestep = 0
        return self.state, {}


## Running the model for 30 days

In [ ]:
ACTION_NAMES = ["study", "attend lecture", "leisure", "socialize", "rest", "skip"]

env = StudentEnv()
obs = env.reset()

lines = []

for day in range(30):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    
    energy, stress, self_efficacy, social_need, days_until_deadline = obs
    
    line = (
        f"Day {day + 1:02d} | "
        f"Action: {ACTION_NAMES[action]:<16} | "
        f"energy: {energy:.2f} | "
        f"stress: {stress:.2f} | "
        f"self_efficacy: {self_efficacy:.2f} | "
        f"social_need: {social_need:.2f} | "
        f"days_until_deadline: {int(days_until_deadline)} | "
        f"reward: {reward:+.2f}"
    )
    lines.append(line)
    
    if done:
        break

with open("student_states.txt", "w") as f:
    f.write("\n".join(lines))

print("Done — saved to student_states.txt")

ValueError: too many values to unpack (expected 4)